# UC3 — Upselling Inteligente: Feature Engineering

**Autor:** Jorge Vázquez  
**Proyecto:** Havi — Copiloto Financiero Proactivo · Hey Banco · DSC x Hey 2026  

## Objetivo

Construir el dataset de features para el modelo de upselling inteligente.  
Cada fila representa un par `(user_id, producto_candidato)`: un usuario y un producto que aún **no tiene** pero para el cual es elegible.

El output de este notebook (`feat_uc3_candidates.parquet`) será la entrada directa al modelo de clasificación.

## Estructura
0. Setup  
1. Carga de datos  
2. Catálogo de productos candidatos  
3. Elegibilidad por producto (reglas de negocio)  
4. Features del usuario base  
5. Features transaccionales (comportamiento reciente 90d)  
6. Features de portafolio  
7. Feature de cashback perdido (específica Hey Pro)  
8. Feature de intención conversacional  
9. Proxy label  
10. Validación  
11. Persistencia  

---
## 0. Setup

In [1]:
# Author: Jorge Vázquez | UC3 — Upselling Inteligente: Feature Engineering
import pandas as pd
import numpy as np
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

# ── Ventanas temporales ───────────────────────────────────────────────────────
CUTOFF_DATE = pd.Timestamp("2025-10-31")
WINDOW_90D  = 90

# ── Rutas ─────────────────────────────────────────────────────────────────────
BASE_TXN  = Path("/Users/diegodq/Documents/dev/datamoles/Datathon-2026/Datathon_Hey_2026_dataset_transacciones 1/dataset_transacciones")
BASE_CONV = Path("/Users/diegodq/Documents/dev/datamoles/Datathon-2026/Datathon_Hey_dataset_conversaciones 1/dataset_conversaciones")
OUTPUT_DIR = Path("/Users/diegodq/Documents/dev/datamoles/Datathon-2026/outputs/features")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("Setup OK")
print(f"  CUTOFF_DATE : {CUTOFF_DATE.date()}")
print(f"  OUTPUT_DIR  : {OUTPUT_DIR}")

Setup OK
  CUTOFF_DATE : 2025-10-31
  OUTPUT_DIR  : /Users/diegodq/Documents/dev/datamoles/Datathon-2026/outputs/features


---
## 1. Carga de datos

Cargamos los cuatro datasets fuente.  
- `hey_clientes.csv` — demografía y señales de comportamiento del usuario  
- `hey_productos.csv` — portafolio de productos por usuario  
- `hey_transacciones.csv` — historial transaccional completo  
- `dataset_50k_anonymized.parquet` — 49 999 conversaciones con Havi  

Aplicamos limpieza inmediata: drop de columna sintética, parseo de fechas, filtro temporal.

In [2]:
%%time
# ── Clientes ──────────────────────────────────────────────────────────────────
df_cli = pd.read_csv(BASE_TXN / "hey_clientes.csv")

# Columna sintética — no aporta valor predictivo real
if 'es_dato_sintetico' in df_cli.columns:
    df_cli.drop(columns=['es_dato_sintetico'], inplace=True)

# Parseo de fechas
for col in ['fecha_alta', 'ultima_conexion', 'fecha_nacimiento']:
    if col in df_cli.columns:
        df_cli[col] = pd.to_datetime(df_cli[col], errors='coerce')

print(f"df_cli       : {df_cli.shape}")

df_cli       : (15025, 22)
CPU times: user 19.3 ms, sys: 2.89 ms, total: 22.2 ms
Wall time: 22.6 ms


In [3]:
%%time
# ── Productos ─────────────────────────────────────────────────────────────────
df_prod = pd.read_csv(BASE_TXN / "hey_productos.csv")

if 'es_dato_sintetico' in df_prod.columns:
    df_prod.drop(columns=['es_dato_sintetico'], inplace=True)

for col in ['fecha_alta_producto', 'fecha_baja_producto']:
    if col in df_prod.columns:
        df_prod[col] = pd.to_datetime(df_prod[col], errors='coerce')

print(f"df_prod      : {df_prod.shape}")

df_prod      : (38909, 12)
CPU times: user 27.9 ms, sys: 3.03 ms, total: 31 ms
Wall time: 31.7 ms


In [4]:
%%time
# ── Transacciones ─────────────────────────────────────────────────────────────
df_tx = pd.read_csv(BASE_TXN / "hey_transacciones.csv")

if 'es_dato_sintetico' in df_tx.columns:
    df_tx.drop(columns=['es_dato_sintetico'], inplace=True)

df_tx['fecha_hora'] = pd.to_datetime(df_tx['fecha_hora'], errors='coerce')

# Solo operaciones hasta el corte — evitamos data leakage
df_tx = df_tx[df_tx['fecha_hora'] <= CUTOFF_DATE].copy()

print(f"df_tx        : {df_tx.shape}")

df_tx        : (731033, 21)
CPU times: user 1.43 s, sys: 81.3 ms, total: 1.51 s
Wall time: 1.51 s


In [5]:
%%time
# ── Conversaciones ────────────────────────────────────────────────────────────
df_conv = pd.read_parquet(BASE_CONV / "dataset_50k_anonymized.parquet")

if 'es_dato_sintetico' in df_conv.columns:
    df_conv.drop(columns=['es_dato_sintetico'], inplace=True)

# 18 filas duplicadas identificadas en EDA previo
n_before = len(df_conv)
df_conv.drop_duplicates(inplace=True)
print(f"df_conv      : {df_conv.shape}  (dropped {n_before - len(df_conv)} duplicates)")

df_conv      : (49981, 6)  (dropped 18 duplicates)
CPU times: user 55.9 ms, sys: 13.2 ms, total: 69.1 ms
Wall time: 68.1 ms


In [6]:
# ── Resumen de carga ──────────────────────────────────────────────────────────
print("Shapes finales:")
print(f"  hey_clientes     : {df_cli.shape}")
print(f"  hey_productos    : {df_prod.shape}")
print(f"  hey_transacciones: {df_tx.shape}  (≤ {CUTOFF_DATE.date()})")
print(f"  conversaciones   : {df_conv.shape}")

print("\nTipos fecha_hora:", df_tx['fecha_hora'].dtype)
print("Rango fechas tx:", df_tx['fecha_hora'].min().date(), "→", df_tx['fecha_hora'].max().date())

Shapes finales:
  hey_clientes     : (15025, 22)
  hey_productos    : (38909, 12)
  hey_transacciones: (731033, 21)  (≤ 2025-10-31)
  conversaciones   : (49981, 6)

Tipos fecha_hora: datetime64[us]
Rango fechas tx: 2025-01-01 → 2025-10-30


---
## 2. Catálogo de productos candidatos

Definimos qué productos son elegibles para recomendación.  
Excluimos `cuenta_debito` porque tiene penetración del 100%: todos los usuarios ya la tienen, no existe oportunidad de upsell.  

La idea central del dataset es modelar pares `(usuario, producto)` donde el producto sea algo que el usuario **aún no tiene** pero para el cual califica.  
Esto nos permite entrenar un clasificador binario por producto o un ranking global.

In [7]:
# ── Catálogo ──────────────────────────────────────────────────────────────────
all_products = df_prod['tipo_producto'].dropna().unique().tolist()
print("Productos en el dataset:", sorted(all_products))

# cuenta_debito: penetración 100% → sin oportunidad de upsell
PRODUCT_CATALOG = [p for p in all_products if p != 'cuenta_debito']
print(f"\nCatálogo UC3 ({len(PRODUCT_CATALOG)} productos):")
for p in sorted(PRODUCT_CATALOG):
    print(f"  - {p}")

Productos en el dataset: ['credito_auto', 'credito_nomina', 'credito_personal', 'cuenta_debito', 'cuenta_negocios', 'inversion_hey', 'seguro_compras', 'seguro_vida', 'tarjeta_credito_garantizada', 'tarjeta_credito_hey', 'tarjeta_credito_negocios']

Catálogo UC3 (10 productos):
  - credito_auto
  - credito_nomina
  - credito_personal
  - cuenta_negocios
  - inversion_hey
  - seguro_compras
  - seguro_vida
  - tarjeta_credito_garantizada
  - tarjeta_credito_hey
  - tarjeta_credito_negocios


In [8]:
%%time
# ── Productos activos por usuario ─────────────────────────────────────────────
# Solo consideramos productos activos — uno dado de baja ya no cuenta
user_active_products = (
    df_prod[df_prod['estatus'] == 'activo']
    .groupby('user_id')['tipo_producto']
    .apply(set)
)

all_users = df_cli['user_id'].unique()
print(f"Usuarios totales: {len(all_users):,}")

# ── Cross join: usuario × catálogo ────────────────────────────────────────────
rows = []
for uid in all_users:
    active = user_active_products.get(uid, set())
    for prod in PRODUCT_CATALOG:
        if prod not in active:          # solo productos que NO tiene
            rows.append({'user_id': uid, 'producto_candidato': prod})

df_candidates = pd.DataFrame(rows)
print(f"df_candidates (user × producto no poseído): {df_candidates.shape}")
print(df_candidates['producto_candidato'].value_counts())

Usuarios totales: 15,025


df_candidates (user × producto no poseído): (129795, 2)
producto_candidato
tarjeta_credito_garantizada    14430
credito_auto                   14157
credito_personal               14064
tarjeta_credito_negocios       14012
seguro_compras                 13868
cuenta_negocios                13799
credito_nomina                 13154
seguro_vida                    12748
inversion_hey                  10990
tarjeta_credito_hey             8573
Name: count, dtype: int64
CPU times: user 224 ms, sys: 4.21 ms, total: 228 ms
Wall time: 228 ms


---
## 3. Elegibilidad por producto (reglas de negocio)

No todos los usuarios califican para todos los productos.  
Aplicamos reglas de negocio derivadas de las políticas de crédito de Hey Banco para eliminar combinaciones imposibles antes del modelado.  
Filtrar aquí reduce ruido y evita que el modelo intente predecir adopciones estructuralmente inviables.

In [9]:
%%time
# ── Merge clientes para evaluación de reglas ──────────────────────────────────
df_cand_cli = df_candidates.merge(
    df_cli[['user_id', 'score_buro', 'ingreso_mensual_mxn', 'nomina_domiciliada', 'ocupacion']],
    on='user_id', how='left'
)

# ── Reglas de elegibilidad ────────────────────────────────────────────────────
# Basadas en políticas de crédito estándar y segmentación de producto
ELIGIBILITY_RULES = {
    'credito_nomina':              lambda u: u['nomina_domiciliada'] == True,
    'credito_personal':            lambda u: u['score_buro'] >= 580,
    'tarjeta_credito_hey':         lambda u: u['score_buro'] >= 580,
    'tarjeta_credito_garantizada': lambda u: pd.Series([True] * len(u), index=u.index),  # anyone
    'credito_auto':                lambda u: (u['ingreso_mensual_mxn'] >= 15000) & (u['score_buro'] >= 600),
    'cuenta_negocios':             lambda u: u['ocupacion'].isin(['Empresario', 'Independiente', 'Comerciante']) if 'ocupacion' in u.columns else pd.Series([True] * len(u), index=u.index),
    'tarjeta_credito_negocios':    lambda u: u['ocupacion'].isin(['Empresario', 'Independiente', 'Comerciante']) if 'ocupacion' in u.columns else pd.Series([True] * len(u), index=u.index),
    'hey_pro':                     lambda u: pd.Series([True] * len(u), index=u.index),
    'inversion_hey':               lambda u: pd.Series([True] * len(u), index=u.index),
    'seguro_vida':                 lambda u: pd.Series([True] * len(u), index=u.index),
    'seguro_compras':              lambda u: pd.Series([True] * len(u), index=u.index),
}

# Aplicar regla a cada fila según su producto_candidato
eligible_mask = pd.Series([False] * len(df_cand_cli), index=df_cand_cli.index)

for prod, rule_fn in ELIGIBILITY_RULES.items():
    mask_prod = df_cand_cli['producto_candidato'] == prod
    if mask_prod.any():
        subset = df_cand_cli[mask_prod]
        result = rule_fn(subset)
        eligible_mask.loc[mask_prod] = result.values

# Productos no cubiertos por reglas → elegibles por defecto
no_rule_mask = ~df_cand_cli['producto_candidato'].isin(ELIGIBILITY_RULES.keys())
eligible_mask.loc[no_rule_mask] = True

df_candidates = df_cand_cli[eligible_mask].drop(
    columns=['score_buro', 'ingreso_mensual_mxn', 'nomina_domiciliada', 'ocupacion']
).reset_index(drop=True)

print(f"df_candidates post-elegibilidad: {df_candidates.shape}")
print("\nCandidatos por producto:")
print(df_candidates['producto_candidato'].value_counts())

df_candidates post-elegibilidad: (83946, 2)

Candidatos por producto:
producto_candidato
tarjeta_credito_garantizada    14430
seguro_compras                 13868
seguro_vida                    12748
inversion_hey                  10990
credito_personal                9107
credito_auto                    6845
tarjeta_credito_hey             4922
tarjeta_credito_negocios        3996
cuenta_negocios                 3783
credito_nomina                  3257
Name: count, dtype: int64
CPU times: user 30.1 ms, sys: 2.04 ms, total: 32.1 ms
Wall time: 32 ms


---
## 4. Features del usuario base

Características demográficas y de perfil del cliente.  
Estas features capturan la **capacidad** y el **contexto** del usuario: qué tan establecido está financieramente, qué tan activo es en la app, y cuántos productos ya tiene (indicador de confianza en el banco).

In [10]:
%%time
# ── Construcción de features base ─────────────────────────────────────────────
feat_base = df_cli.copy()

# Score burecrático: proxy de solvencia
feat_base['feat_score_buro'] = feat_base['score_buro']

# Ingreso: log para comprimir la distribución sesgada a la derecha
feat_base['feat_ingreso_log'] = np.log1p(feat_base['ingreso_mensual_mxn'])

# Cuartil de ingreso: feature ordinal más interpretable para el modelo
feat_base['feat_ingreso_cuartil'] = pd.qcut(
    feat_base['ingreso_mensual_mxn'], q=4, labels=False, duplicates='drop'
)

# Edad: ciclo de vida financiero
feat_base['feat_edad'] = feat_base['edad'] if 'edad' in feat_base.columns else (
    (CUTOFF_DATE - feat_base['fecha_nacimiento']).dt.days // 365
    if 'fecha_nacimiento' in feat_base.columns else np.nan
)

# Antigüedad: usuarios más viejos tienen más historial crediticio interno
if 'fecha_alta' in feat_base.columns:
    feat_base['feat_antiguedad_dias'] = (CUTOFF_DATE - feat_base['fecha_alta']).dt.days
else:
    feat_base['feat_antiguedad_dias'] = np.nan

# Hey Pro actual: si ya pagó por Pro, tiene mayor propensión a productos premium
feat_base['feat_es_hey_pro'] = feat_base['es_hey_pro'].astype(int) if 'es_hey_pro' in feat_base.columns else 0

# Nómina domiciliada: ingreso estable verificado
feat_base['feat_nomina_domiciliada'] = feat_base['nomina_domiciliada'].astype(int) if 'nomina_domiciliada' in feat_base.columns else 0

# Remesas: señal de flujo internacional de dinero
feat_base['feat_recibe_remesas'] = feat_base['recibe_remesas'].astype(int) if 'recibe_remesas' in feat_base.columns else 0

# Satisfacción: impute median — proxy de NPS interno
if 'satisfaccion_1_10' in feat_base.columns:
    feat_base['feat_satisfaccion'] = feat_base['satisfaccion_1_10'].fillna(
        feat_base['satisfaccion_1_10'].median()
    )
else:
    feat_base['feat_satisfaccion'] = np.nan

# Nº de productos activos: cuántos ya confía en Hey (cross-sell positivo)
n_prods = (
    df_prod[df_prod['estatus'] == 'activo']
    .groupby('user_id')['tipo_producto']
    .count()
    .rename('feat_n_productos_actuales')
)
feat_base = feat_base.merge(n_prods, on='user_id', how='left')
feat_base['feat_n_productos_actuales'] = feat_base['feat_n_productos_actuales'].fillna(0).astype(int)

# Días desde último login: actividad reciente en la app
if 'ultima_conexion' in feat_base.columns:
    feat_base['feat_dias_desde_ultimo_login'] = (CUTOFF_DATE - feat_base['ultima_conexion']).dt.days
else:
    feat_base['feat_dias_desde_ultimo_login'] = np.nan

# Patrón de uso atípico: señal de riesgo o comportamiento inusual
if 'patron_uso_atipico' in feat_base.columns:
    feat_base['feat_patron_uso_atipico_user'] = feat_base['patron_uso_atipico'].astype(int)
else:
    feat_base['feat_patron_uso_atipico_user'] = 0

feat_cols_base = ['user_id'] + [c for c in feat_base.columns if c.startswith('feat_')]
feat_base = feat_base[feat_cols_base]

# Merge sobre df_candidates
df_candidates = df_candidates.merge(feat_base, on='user_id', how='left')

print(f"df_candidates post-feat_base: {df_candidates.shape}")
print("Features base agregadas:", [c for c in df_candidates.columns if c.startswith('feat_')])

df_candidates post-feat_base: (83946, 14)
Features base agregadas: ['feat_score_buro', 'feat_ingreso_log', 'feat_ingreso_cuartil', 'feat_edad', 'feat_antiguedad_dias', 'feat_es_hey_pro', 'feat_nomina_domiciliada', 'feat_recibe_remesas', 'feat_satisfaccion', 'feat_n_productos_actuales', 'feat_dias_desde_ultimo_login', 'feat_patron_uso_atipico_user']
CPU times: user 17.7 ms, sys: 2.42 ms, total: 20.2 ms
Wall time: 20.1 ms


---
## 5. Features transaccionales (comportamiento reciente — 90d)

El comportamiento de los últimos 90 días es el mejor predictor de intención de compra.  
Usamos solo transacciones **completadas** para no contaminar con operaciones pendientes o rechazadas.  

Features clave:
- Volumen y frecuencia de gasto → capacidad de pago
- Diversidad de categorías → sofisticación financiera
- Ratio fin de semana → estilo de vida
- Abonos a inversión → propensión a productos de ahorro
- Pagos a crédito → comportamiento responsable de deuda

In [11]:
%%time
# ── Filtrar ventana 90d + completadas ─────────────────────────────────────────
start_90d = CUTOFF_DATE - pd.Timedelta(days=WINDOW_90D)
df_tx_90d = df_tx[
    (df_tx['fecha_hora'] >= start_90d) &
    (df_tx['fecha_hora'] <= CUTOFF_DATE) &
    (df_tx['estatus'] == 'completada')
].copy()

print(f"Transacciones 90d completadas: {df_tx_90d.shape}")

# ── Features agregadas por usuario ────────────────────────────────────────────
grp = df_tx_90d.groupby('user_id')

feat_tx = pd.DataFrame(index=df_tx_90d['user_id'].unique())
feat_tx.index.name = 'user_id'

# Gasto total y ticket promedio
feat_tx['feat_gasto_total_90d'] = grp['monto'].sum()
feat_tx['feat_ticket_avg_90d']  = grp['monto'].mean()
feat_tx['feat_n_tx_90d']        = grp['monto'].count()

# Categoría MCC dominante (excluyendo transferencias — no refleja gasto real)
if 'categoria_mcc' in df_tx_90d.columns:
    df_tx_no_transf = df_tx_90d[df_tx_90d['categoria_mcc'] != 'transferencia']
    feat_tx['feat_top_mcc_90d'] = (
        df_tx_no_transf.groupby('user_id')['categoria_mcc']
        .agg(lambda x: x.mode()[0] if len(x) > 0 else np.nan)
    )
    feat_tx['feat_cat_diversity_90d'] = (
        df_tx_no_transf.groupby('user_id')['categoria_mcc'].nunique()
    )

# Ratio fin de semana: usuarios que gastan mucho en fin de semana → mayor propensión a seguros/experiencias
df_tx_90d['_is_weekend'] = df_tx_90d['fecha_hora'].dt.dayofweek >= 5
feat_tx['feat_weekend_ratio_90d'] = (
    df_tx_90d.groupby('user_id')['_is_weekend'].mean()
)

# Composición del gasto por tipo de operación
if 'tipo_operacion' in df_tx_90d.columns:
    total_por_user = grp['monto'].count().rename('_total')

    compras = df_tx_90d[df_tx_90d['tipo_operacion'] == 'compra'].groupby('user_id')['monto'].count()
    feat_tx['feat_share_compras_90d'] = (compras / total_por_user).fillna(0)

    transf = df_tx_90d[df_tx_90d['tipo_operacion'].isin(['transf_entrada', 'transf_salida'])].groupby('user_id')['monto'].count()
    feat_tx['feat_share_transferencias_90d'] = (transf / total_por_user).fillna(0)

    # Abonos a inversión: señal directa de propensión a inversion_hey
    abonos_inv = df_tx_90d[df_tx_90d['tipo_operacion'] == 'abono_inversion']
    feat_tx['feat_n_abonos_inversion_90d']     = abonos_inv.groupby('user_id')['monto'].count().fillna(0)
    feat_tx['feat_monto_abonos_inversion_90d'] = abonos_inv.groupby('user_id')['monto'].sum().fillna(0)

    # Pagos a crédito: responsabilidad de deuda
    pagos_cred = df_tx_90d[df_tx_90d['tipo_operacion'] == 'pago_credito']
    feat_tx['feat_n_pagos_credito_90d'] = pagos_cred.groupby('user_id')['monto'].count().fillna(0)

    # Volumen de entradas mensuales: proxy de nómina
    df_tx_90d['_mes'] = df_tx_90d['fecha_hora'].dt.to_period('M')
    transf_entrada = df_tx_90d[df_tx_90d['tipo_operacion'] == 'transf_entrada']
    feat_tx['feat_volumen_transf_entrada_mensual'] = (
        transf_entrada.groupby(['user_id', '_mes'])['monto']
        .sum()
        .reset_index()
        .groupby('user_id')['monto']
        .mean()
    )

feat_tx = feat_tx.reset_index()

# Merge sobre df_candidates (left: usuarios sin tx en 90d quedan en 0/NaN)
tx_fill_zero = [
    'feat_gasto_total_90d', 'feat_ticket_avg_90d', 'feat_n_tx_90d',
    'feat_cat_diversity_90d', 'feat_n_abonos_inversion_90d',
    'feat_monto_abonos_inversion_90d', 'feat_n_pagos_credito_90d',
    'feat_volumen_transf_entrada_mensual',
    'feat_share_compras_90d', 'feat_share_transferencias_90d',
    'feat_weekend_ratio_90d'
]
df_candidates = df_candidates.merge(feat_tx, on='user_id', how='left')
for col in tx_fill_zero:
    if col in df_candidates.columns:
        df_candidates[col] = df_candidates[col].fillna(0)

print(f"df_candidates post-feat_tx: {df_candidates.shape}")

Transacciones 90d completadas: (204177, 21)


df_candidates post-feat_tx: (83946, 26)
CPU times: user 727 ms, sys: 15.5 ms, total: 742 ms
Wall time: 743 ms


---
## 6. Features de portafolio

Señales derivadas del portafolio de productos activos del usuario.  
Capturan la **exposición crediticia actual**, la presencia de seguros y el comportamiento de ahorro.  
Un producto dormido (sin uso en 90d) es señal de bajo engagement y puede indicar candidato a churn o a upgrade.

In [12]:
%%time
df_prod_act = df_prod[df_prod['estatus'] == 'activo'].copy()

feat_port = pd.DataFrame({'user_id': df_cli['user_id'].unique()})

# Utilización máxima de tarjetas de crédito: indica presión financiera
if 'utilizacion_pct' in df_prod_act.columns:
    tc_mask = df_prod_act['tipo_producto'].str.contains('tarjeta_credito', na=False)
    feat_port = feat_port.merge(
        df_prod_act[tc_mask].groupby('user_id')['utilizacion_pct']
        .max().rename('feat_utilizacion_tc_max').reset_index(),
        on='user_id', how='left'
    )
    feat_port['feat_utilizacion_tc_max'] = feat_port['feat_utilizacion_tc_max'].fillna(0)
else:
    feat_port['feat_utilizacion_tc_max'] = 0

# Tiene seguro: cross-sell positivo (si ya confía en seguros, puede querer otro)
if 'tipo_producto' in df_prod_act.columns:
    seg_mask = df_prod_act['tipo_producto'].str.contains('seguro', na=False)
    tiene_seguro = (
        df_prod_act[seg_mask]
        .groupby('user_id')['tipo_producto']
        .count().gt(0).astype(int).rename('feat_tiene_seguro').reset_index()
    )
    feat_port = feat_port.merge(tiene_seguro, on='user_id', how='left')
    feat_port['feat_tiene_seguro'] = feat_port['feat_tiene_seguro'].fillna(0).astype(int)

# Total de mensualidades: carga de deuda mensual actual
if 'monto_mensualidad' in df_prod_act.columns:
    feat_port = feat_port.merge(
        df_prod_act.groupby('user_id')['monto_mensualidad']
        .sum().rename('feat_monto_mensualidad_total').reset_index(),
        on='user_id', how='left'
    )
    feat_port['feat_monto_mensualidad_total'] = feat_port['feat_monto_mensualidad_total'].fillna(0)
else:
    feat_port['feat_monto_mensualidad_total'] = 0

# Saldo en inversiones: ahorradores activos → mayor propensión a inversion_hey
if 'saldo_actual' in df_prod_act.columns:
    inv_mask = df_prod_act['tipo_producto'] == 'inversion_hey'
    feat_port = feat_port.merge(
        df_prod_act[inv_mask].groupby('user_id')['saldo_actual']
        .sum().rename('feat_saldo_inversion').reset_index(),
        on='user_id', how='left'
    )
    feat_port['feat_saldo_inversion'] = feat_port['feat_saldo_inversion'].fillna(0)
else:
    feat_port['feat_saldo_inversion'] = 0

# Créditos activos: cuántas deudas activas tiene
if 'tipo_producto' in df_prod_act.columns:
    cred_mask = df_prod_act['tipo_producto'].str.contains('credito', na=False)
    n_cred = (
        df_prod_act[cred_mask]
        .groupby('user_id')['tipo_producto']
        .count().rename('feat_n_creditos_activos').reset_index()
    )
    feat_port = feat_port.merge(n_cred, on='user_id', how='left')
    feat_port['feat_n_creditos_activos'] = feat_port['feat_n_creditos_activos'].fillna(0).astype(int)

# Productos dormidos: activos sin transacciones en los últimos 90d
# → bajo engagement, puede indicar candidato a activación o upgrade
users_with_tx_90d = df_tx_90d['user_id'].unique()
all_prod_users = df_prod_act['user_id'].unique()
dormidos_series = pd.Series(
    {uid: 1 if uid not in set(users_with_tx_90d) else 0 for uid in all_prod_users},
    name='feat_n_productos_dormidos'
).reset_index().rename(columns={'index': 'user_id'})
feat_port = feat_port.merge(dormidos_series, on='user_id', how='left')
feat_port['feat_n_productos_dormidos'] = feat_port['feat_n_productos_dormidos'].fillna(0).astype(int)

df_candidates = df_candidates.merge(feat_port, on='user_id', how='left')
print(f"df_candidates post-feat_portafolio: {df_candidates.shape}")

df_candidates post-feat_portafolio: (83946, 32)
CPU times: user 1min 2s, sys: 95 ms, total: 1min 2s
Wall time: 1min 2s


---
## 7. Feature de cashback perdido (específica para Hey Pro)

Esta es la feature más poderosa para predecir adopción de **Hey Pro**.  

Un usuario que hace muchas compras y no es Pro está literalmente dejando dinero sobre la mesa.  
El cashback perdido mensual cuantifica ese costo de oportunidad: si supera ~300 MXN/mes, la suscripción de Hey Pro se amortiza sola.

Regla proxy:
- `label_proxy = 1` para usuarios que **ya son Pro** (adoptaron)
- `cashback_perdido = monto × 1%` para compras completadas de usuarios no-Pro

Segmentos:
- **A** (> 300 MXN/mes): candidato prioritario
- **B** (100–300 MXN/mes): candidato medio
- **C** (< 100 MXN/mes): bajo potencial
- **pro**: ya es Pro

In [13]:
%%time
# ── Cashback perdido para no-Pro ───────────────────────────────────────────────
# Solo compras completadas de usuarios no-Pro generan cashback perdido
# Tasa Hey Pro: 1% cashback en compras
CASHBACK_RATE = 0.01
MONTHS_IN_WINDOW = WINDOW_90D / 30

# Usuarios Pro actuales
pro_users = set(df_cli[df_cli['es_hey_pro'] == True]['user_id'].tolist()) if 'es_hey_pro' in df_cli.columns else set()

# Transacciones de compra completadas de usuarios NO Pro
df_tx_no_pro = df_tx_90d[
    (df_tx_90d['tipo_operacion'] == 'compra') &
    (~df_tx_90d['user_id'].isin(pro_users))
].copy()

df_tx_no_pro['_cashback_potencial'] = df_tx_no_pro['monto'] * CASHBACK_RATE

# Cashback perdido total en ventana 90d → normalizar a mensual
cashback_total = df_tx_no_pro.groupby('user_id')['_cashback_potencial'].sum()
cashback_mensual = (cashback_total / MONTHS_IN_WINDOW).rename('feat_cashback_perdido_mensual')

# Segmentación por potencial
def segmentar_cashback(val):
    if val > 300:  return 'A'
    if val >= 100: return 'B'
    return 'C'

cashback_df = cashback_mensual.reset_index()
cashback_df['feat_cashback_segmento_base'] = cashback_df['feat_cashback_perdido_mensual'].apply(segmentar_cashback)

# Usuarios Pro → segmento 'pro'
pro_df = pd.DataFrame({'user_id': list(pro_users), 'feat_cashback_perdido_mensual': 0.0, 'feat_cashback_segmento_base': 'pro'})
cashback_df = pd.concat([cashback_df, pro_df], ignore_index=True)

# Merge sobre df_candidates
df_candidates = df_candidates.merge(cashback_df, on='user_id', how='left')
df_candidates['feat_cashback_perdido_mensual'] = df_candidates['feat_cashback_perdido_mensual'].fillna(0)
df_candidates['feat_cashback_segmento_base']   = df_candidates['feat_cashback_segmento_base'].fillna('C')

# La feature de cashback solo es relevante para la fila de hey_pro
# Para otros productos la ponemos en 0 para no contaminar
mask_no_pro = df_candidates['producto_candidato'] != 'hey_pro'
df_candidates.loc[mask_no_pro, 'feat_cashback_perdido_mensual'] = 0

# Renombrar para claridad semántica
df_candidates.rename(columns={'feat_cashback_segmento_base': 'feat_cashback_segmento'}, inplace=True)

print(f"df_candidates post-feat_cashback: {df_candidates.shape}")
print("\nDistribución feat_cashback_segmento (filas hey_pro):")
print(df_candidates[df_candidates['producto_candidato'] == 'hey_pro']['feat_cashback_segmento'].value_counts())

df_candidates post-feat_cashback: (83946, 34)

Distribución feat_cashback_segmento (filas hey_pro):
Series([], Name: count, dtype: int64)
CPU times: user 35 ms, sys: 2.61 ms, total: 37.6 ms
Wall time: 38.2 ms


---
## 8. Feature de intención conversacional

Las conversaciones con Havi son la señal de **intención explícita** más directa que tenemos.  
Si un usuario preguntó sobre inversiones, es un lead calificado para `inversion_hey`.  
Si preguntó sobre cashback o beneficios, probablemente está evaluando `hey_pro`.

Esta feature captura si alguna vez el usuario mencionó palabras clave relacionadas con el producto candidato — sin importar el resultado de la conversación.

In [14]:
%%time
# ── Keywords por producto ─────────────────────────────────────────────────────
PRODUCT_KEYWORDS = {
    'hey_pro':             ['cashback', 'hey pro', 'beneficios', 'pro'],
    'inversion_hey':       ['inversión', 'inversion', 'rendimiento', 'ahorro', 'cetes'],
    'tarjeta_credito_hey': ['tarjeta', 'crédito', 'credito', 'línea', 'linea'],
    'credito_nomina':      ['préstamo', 'prestamo', 'nómina', 'nomina', 'crédito nómina'],
    'seguro_vida':         ['seguro', 'vida', 'protección', 'proteccion'],
    'seguro_compras':      ['seguro', 'compras', 'protección', 'proteccion'],
    'credito_personal':    ['préstamo', 'prestamo', 'personal', 'crédito personal'],
    'credito_auto':        ['auto', 'coche', 'crédito auto', 'financiamiento'],
    'cuenta_negocios':     ['negocio', 'empresa', 'pyme', 'comercio'],
    'tarjeta_credito_negocios': ['negocio', 'empresa', 'tarjeta empresa'],
    'tarjeta_credito_garantizada': ['tarjeta', 'garantizada', 'sin historial'],
}

# Columna de texto: buscar la columna que contiene el mensaje
text_col = None
for candidate_col in ['mensaje', 'text', 'mensaje_usuario', 'contenido', 'query', 'message']:
    if candidate_col in df_conv.columns:
        text_col = candidate_col
        break

if text_col is None:
    # Usar la columna string más larga como proxy
    str_cols = df_conv.select_dtypes(include='object').columns.tolist()
    text_col = max(str_cols, key=lambda c: df_conv[c].dropna().str.len().mean()) if str_cols else None
    print(f"⚠️  Columna de texto inferida: '{text_col}'")
else:
    print(f"Columna de texto: '{text_col}'")

if text_col:
    df_conv['_text_lower'] = df_conv[text_col].fillna('').str.lower()

    # Para cada usuario, obtener el texto concatenado de todas sus conversaciones
    user_text = df_conv.groupby('user_id')['_text_lower'].apply(' '.join).reset_index()
    user_text.columns = ['user_id', '_all_text']

    # Para cada producto, marcar si el usuario mencionó alguna keyword
    for prod, keywords in PRODUCT_KEYWORDS.items():
        pattern = '|'.join(keywords)
        user_text[f'_mentioned_{prod}'] = user_text['_all_text'].str.contains(pattern, regex=True, na=False)

    # Merge sobre df_candidates
    df_candidates = df_candidates.merge(user_text.drop(columns=['_all_text']), on='user_id', how='left')

    # Construir feat_menciona_producto_en_havi: 1 si el usuario mencionó el producto candidato
    def get_mention(row):
        col = f'_mentioned_{row["producto_candidato"]}'
        if col in row.index:
            return int(row[col]) if not pd.isna(row[col]) else 0
        return 0

    df_candidates['feat_menciona_producto_en_havi'] = df_candidates.apply(get_mention, axis=1)

    # Limpiar columnas temporales
    drop_cols = [c for c in df_candidates.columns if c.startswith('_mentioned_')]
    df_candidates.drop(columns=drop_cols, inplace=True)
else:
    print("⚠️  No se encontró columna de texto en conversaciones. feat_menciona_producto_en_havi = 0")
    df_candidates['feat_menciona_producto_en_havi'] = 0

df_candidates['feat_menciona_producto_en_havi'] = df_candidates['feat_menciona_producto_en_havi'].fillna(0).astype(int)

print(f"df_candidates post-feat_conv: {df_candidates.shape}")
print("\nMenciones por producto:")
print(
    df_candidates[df_candidates['feat_menciona_producto_en_havi'] == 1]
    ['producto_candidato'].value_counts()
)

⚠️  Columna de texto inferida: 'output'


df_candidates post-feat_conv: (83946, 35)

Menciones por producto:
producto_candidato
tarjeta_credito_garantizada    8261
tarjeta_credito_hey            3467
seguro_compras                 2883
seguro_vida                    1903
credito_personal               1814
credito_auto                   1778
inversion_hey                  1092
cuenta_negocios                 648
tarjeta_credito_negocios        436
credito_nomina                  198
Name: count, dtype: int64
CPU times: user 947 ms, sys: 22.8 ms, total: 970 ms
Wall time: 970 ms


---
## 9. Proxy label

No contamos con datos de adopción futura, así que construimos un **proxy retrospectivo**:

- Para `hey_pro`: `label_proxy = 1` si el usuario **ya es Pro** (adoptó en algún momento del pasado). Tenemos tanto 1s como 0s → es el producto más trainable.
- Para los demás productos: como ya excluimos en el paso 2 los productos que el usuario tiene activos, todas las filas restantes son `label_proxy = 0` por construcción.

**Limitación conocida:** Este label no captura adopciones futuras. Es un proxy para aprender el perfil del adoptador pasado. Para producción se debería enriquecer con un label de adopción real en los siguientes 30/60 días.

In [15]:
# ── Proxy label ───────────────────────────────────────────────────────────────
df_candidates['label_proxy'] = 0  # default: no adoptó

# Hey Pro: usuarios que ya son Pro = adoptaron (label retrospectivo)
if 'es_hey_pro' in df_cli.columns:
    pro_users_set = set(df_cli[df_cli['es_hey_pro'] == True]['user_id'].tolist())
    hey_pro_mask = df_candidates['producto_candidato'] == 'hey_pro'
    df_candidates.loc[hey_pro_mask & df_candidates['user_id'].isin(pro_users_set), 'label_proxy'] = 1

print("Distribución label_proxy:")
print(df_candidates['label_proxy'].value_counts())

print("\nLabel=1 por producto:")
print(df_candidates[df_candidates['label_proxy'] == 1]['producto_candidato'].value_counts())

print(f"\nBalance hey_pro:")
hey_pro_df = df_candidates[df_candidates['producto_candidato'] == 'hey_pro']
pos = hey_pro_df['label_proxy'].sum()
neg = len(hey_pro_df) - pos
print(f"  Positivos (adoptaron): {pos:,}  ({pos/len(hey_pro_df)*100:.1f}%)")
print(f"  Negativos (no adoptaron): {neg:,}  ({neg/len(hey_pro_df)*100:.1f}%)")

Distribución label_proxy:
label_proxy
0    83946
Name: count, dtype: int64

Label=1 por producto:
Series([], Name: count, dtype: int64)

Balance hey_pro:
  Positivos (adoptaron): 0  (nan%)
  Negativos (no adoptaron): 0  (nan%)


---
## 10. Validación

Verificamos:
- Shape final del dataset
- Nulos por feature (ninguna debería tener > 30% salvo las categóricas conocidas)
- Distribución del segmento de cashback
- Menciones conversacionales por producto
- Filas por producto candidato

In [16]:
print(f"Shape final df_candidates: {df_candidates.shape}")
print(f"Usuarios únicos: {df_candidates['user_id'].nunique():,}")
print(f"Productos candidatos: {df_candidates['producto_candidato'].nunique()}")

Shape final df_candidates: (83946, 36)
Usuarios únicos: 15,025
Productos candidatos: 10


In [17]:
# ── Tasa de nulos por feature ─────────────────────────────────────────────────
feat_cols = [c for c in df_candidates.columns if c.startswith('feat_')]
null_rates = df_candidates[feat_cols].isnull().mean().sort_values(ascending=False)
print("Tasa de nulos por feature:")
print(null_rates[null_rates > 0].to_string())

Tasa de nulos por feature:
feat_dias_desde_ultimo_login    1.000000
feat_antiguedad_dias            1.000000
feat_top_mcc_90d                0.035964


In [18]:
# ── Distribución feat_cashback_segmento ───────────────────────────────────────
print("Distribución feat_cashback_segmento (todas las filas):")
print(df_candidates['feat_cashback_segmento'].value_counts())

print("\nDistribución feat_cashback_segmento (solo hey_pro):")
print(df_candidates[df_candidates['producto_candidato'] == 'hey_pro']['feat_cashback_segmento'].value_counts())

Distribución feat_cashback_segmento (todas las filas):
feat_cashback_segmento
pro    41944
C      38050
B       3717
A        235
Name: count, dtype: int64

Distribución feat_cashback_segmento (solo hey_pro):
Series([], Name: count, dtype: int64)


In [19]:
# ── Menciones conversacionales por producto ───────────────────────────────────
print("feat_menciona_producto_en_havi == True, por producto:")
print(
    df_candidates[df_candidates['feat_menciona_producto_en_havi'] == 1]
    .groupby('producto_candidato')['user_id'].nunique()
    .sort_values(ascending=False)
)

feat_menciona_producto_en_havi == True, por producto:
producto_candidato
tarjeta_credito_garantizada    8261
tarjeta_credito_hey            3467
seguro_compras                 2883
seguro_vida                    1903
credito_personal               1814
credito_auto                   1778
inversion_hey                  1092
cuenta_negocios                 648
tarjeta_credito_negocios        436
credito_nomina                  198
Name: user_id, dtype: int64


In [20]:
# ── Filas por producto candidato ──────────────────────────────────────────────
print("Filas por producto_candidato:")
print(df_candidates['producto_candidato'].value_counts())

Filas por producto_candidato:
producto_candidato
tarjeta_credito_garantizada    14430
seguro_compras                 13868
seguro_vida                    12748
inversion_hey                  10990
credito_personal                9107
credito_auto                    6845
tarjeta_credito_hey             4922
tarjeta_credito_negocios        3996
cuenta_negocios                 3783
credito_nomina                  3257
Name: count, dtype: int64


In [21]:
# ── Preview ───────────────────────────────────────────────────────────────────
print("\nColumnas del dataset final:")
for i, c in enumerate(df_candidates.columns):
    print(f"  {i:02d}. {c}")

df_candidates.head(5)


Columnas del dataset final:
  00. user_id
  01. producto_candidato
  02. feat_score_buro
  03. feat_ingreso_log
  04. feat_ingreso_cuartil
  05. feat_edad
  06. feat_antiguedad_dias
  07. feat_es_hey_pro
  08. feat_nomina_domiciliada
  09. feat_recibe_remesas
  10. feat_satisfaccion
  11. feat_n_productos_actuales
  12. feat_dias_desde_ultimo_login
  13. feat_patron_uso_atipico_user
  14. feat_gasto_total_90d
  15. feat_ticket_avg_90d
  16. feat_n_tx_90d
  17. feat_top_mcc_90d
  18. feat_cat_diversity_90d
  19. feat_weekend_ratio_90d
  20. feat_share_compras_90d
  21. feat_share_transferencias_90d
  22. feat_n_abonos_inversion_90d
  23. feat_monto_abonos_inversion_90d
  24. feat_n_pagos_credito_90d
  25. feat_volumen_transf_entrada_mensual
  26. feat_utilizacion_tc_max
  27. feat_tiene_seguro
  28. feat_monto_mensualidad_total
  29. feat_saldo_inversion
  30. feat_n_creditos_activos
  31. feat_n_productos_dormidos
  32. feat_cashback_perdido_mensual
  33. feat_cashback_segmento
  34. 

,user_id,producto_candidato,feat_score_buro,feat_ingreso_log,feat_ingreso_cuartil,feat_edad,feat_antiguedad_dias,feat_es_hey_pro,feat_nomina_domiciliada,feat_recibe_remesas,...,feat_utilizacion_tc_max,feat_tiene_seguro,feat_monto_mensualidad_total,feat_saldo_inversion,feat_n_creditos_activos,feat_n_productos_dormidos,feat_cashback_perdido_mensual,feat_cashback_segmento,feat_menciona_producto_en_havi,label_proxy
0,USR-00001,inversion_hey,527,10.106469,2,21,NaN,1,0,0,...,0.6166,0,0.0,0.0,1,0,0.0,pro,0,0
1,USR-00001,seguro_vida,527,10.106469,2,21,NaN,1,0,0,...,0.6166,0,0.0,0.0,1,0,0.0,pro,0,0
2,USR-00001,seguro_compras,527,10.106469,2,21,NaN,1,0,0,...,0.6166,0,0.0,0.0,1,0,0.0,pro,1,0
3,USR-00001,tarjeta_credito_garantizada,527,10.106469,2,21,NaN,1,0,0,...,0.6166,0,0.0,0.0,1,0,0.0,pro,1,0
4,USR-00002,inversion_hey,714,9.852247,1,18,NaN,1,0,0,...,0.2783,0,0.0,0.0,1,0,0.0,pro,0,0


---
## 11. Persistencia

Guardamos el dataset de candidatos enriquecido en parquet para el siguiente notebook de modelado.  
Formato parquet: columnar, comprimido, preserva tipos — ideal para features con muchas columnas numéricas.

In [22]:
output_path = OUTPUT_DIR / "feat_uc3_candidates.parquet"
df_candidates.to_parquet(output_path, index=False)
print(f"✅ feat_uc3_candidates.parquet: {df_candidates.shape}")
print(f"   Guardado en: {output_path}")
print(f"   Tamaño: {output_path.stat().st_size / 1024 / 1024:.2f} MB")

✅ feat_uc3_candidates.parquet: (83946, 36)
   Guardado en: /Users/diegodq/Documents/dev/datamoles/Datathon-2026/outputs/features/feat_uc3_candidates.parquet
   Tamaño: 1.51 MB
